# dlt → Unity Catalog managed tables (custom write adapter)

This notebook shows how to ingest data with **dlt** and land it as **UC-managed Delta tables** using the platform's custom write adapter pattern — the exact same approach the automated pipeline uses (`data_platform.uc_managed_destination`), but bound to **your own Unity Catalog identity**.

**How governance works here (identical to Superset):**
JupyterHub logs you in via Keycloak; this notebook exchanges *your* token for a Unity Catalog token and binds it to a Spark Connect session. Every write goes through **Spark Connect → UC credential vending** — there is **no static object-store key** and **no ambient admin**. UC enforces *your* grants.

**You must run this as a builder** (e.g. the `engineer` user → `data-engineer` persona, or a team's ingest SA). An `analyst` can read `gold` but will be **denied** writing `bronze` — that denial is the platform working as designed.

> Managed tables are the multi-tenant-safe primitive: UC owns the storage path, so no other principal can register over it (un-hijackable).

## 1. Who am I? (per-user Unity Catalog session)

`uc_notebook.uc_session()` returns a Spark Connect session bound to *your* UC token. `whoami()` shows which layers you can actually read/write.

In [ ]:
import uc_notebook

spark = uc_notebook.uc_session()
# Arrow-native transfer for the dlt batches (no pandas round-trip).
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
uc_notebook.whoami(spark)

## 2. Define a dlt source

Any dlt source works (databases, APIs, files, …). Here we use a tiny in-notebook resource so the demo is self-contained. The resource **name becomes the table name**: `nb_dlt_demo` → `analytics.bronze.nb_dlt_demo`.

Declaring `columns` makes dlt **coerce** the raw values to real types during normalize, so the Arrow batch carries proper types into the managed table.

In [ ]:
import dlt

@dlt.resource(name="nb_dlt_demo", write_disposition="replace", columns={
    "order_id":    {"data_type": "bigint"},
    "customer_id": {"data_type": "bigint"},
    "status":      {"data_type": "text"},
    "amount":      {"data_type": "decimal", "precision": 18, "scale": 2},
})
def nb_dlt_demo():
    for i in range(1, 51):
        yield {
            "order_id": i,
            "customer_id": 1000 + (i % 7),
            "status": "completed" if i % 3 else "pending",
            "amount": round(9.99 * i, 2),
        }

## 3. The custom UC-managed destination

This is a self-contained version of `data_platform.uc_managed_destination`, adapted to write through **your** Spark session. dlt hands each normalized batch as Arrow (`loader_file_format="parquet"`); we build a Spark DataFrame Arrow-natively and write a **managed** Delta table (`writeTo(...).using("delta")`). UC vends per-table storage credentials at commit time — no static key.

Only `replace` (full refresh) and `append` (accumulate) are supported. Writes are serialized per table (single-writer avoids Delta commit conflicts).

In [ ]:
import threading
import pyarrow as pa

def make_uc_managed_destination(spark, catalog="analytics", schema="bronze", batch_size=100_000):
    """dlt custom destination writing UC-managed Delta tables via `spark` (your session)."""
    lock = threading.Lock()
    handled = set()  # tables create/reset-handled this load

    def _to_spark_df(arrow_table):
        try:
            return spark.createDataFrame(arrow_table)                 # Spark 4.x Connect: Arrow Table
        except (TypeError, ValueError, NotImplementedError):
            return spark.createDataFrame(arrow_table.to_pandas())     # Arrow-backed pandas fallback

    @dlt.destination(loader_file_format="parquet", batch_size=batch_size, name="uc_managed")
    def uc_managed(items, table):
        name = table["name"]
        if name.startswith("_dlt"):          # skip dlt bookkeeping tables
            return
        disp = table.get("write_disposition", "append")
        if disp not in ("replace", "append"):
            raise ValueError(f"uc_managed supports only replace/append, got {disp!r}")

        if isinstance(items, pa.Table):
            arrow = items
        elif isinstance(items, pa.RecordBatch):
            arrow = pa.Table.from_batches([items])
        elif isinstance(items, list) and items and isinstance(items[0], pa.RecordBatch):
            arrow = pa.Table.from_batches(items)
        else:
            arrow = pa.Table.from_pylist(list(items))

        full = f"{catalog}.{schema}.{name}"
        with lock:                            # one writer per table = no commit conflicts
            sdf = _to_spark_df(arrow)
            first = full not in handled
            writer = sdf.writeTo(full).using("delta")
            if disp == "replace":
                writer.createOrReplace() if first else writer.append()
            else:  # append
                if first and not spark.catalog.tableExists(full):
                    writer.create()
                else:
                    writer.append()
            handled.add(full)
            print(f"  wrote {arrow.num_rows} rows -> {full} ({disp})")

    return uc_managed

## 4. Run the pipeline

`LOAD__WORKERS=1` keeps a single writer per managed table (dlt's extract/normalize parallelism is unaffected). The write lands `analytics.bronze.nb_dlt_demo` — which requires **bronze write** (data-engineer / ingest SA). As an `analyst` this cell will raise `PERMISSION_DENIED`.

In [ ]:
import os
os.environ["LOAD__WORKERS"] = "1"

pipeline = dlt.pipeline(
    pipeline_name="nb_dlt_demo",
    destination=make_uc_managed_destination(spark, catalog="analytics", schema="bronze"),
    dataset_name="bronze",
    pipelines_dir="/tmp/dlt_nb",
)

info = pipeline.run(nb_dlt_demo(), write_disposition="replace")
print(info)

## 5. Read it back (as you, through UC)

In [ ]:
df = spark.sql("SELECT * FROM analytics.bronze.nb_dlt_demo ORDER BY order_id LIMIT 10")
df.show()
print("total rows:", spark.sql("SELECT count(*) FROM analytics.bronze.nb_dlt_demo").collect()[0][0])

# Confirm it is a MANAGED table (UC owns the storage path):
spark.sql("DESCRIBE EXTENDED analytics.bronze.nb_dlt_demo").show(truncate=False)

## 6. Clean up the demo table

Optional — drop the demo table so you don't leave it behind.

In [ ]:
spark.sql("DROP TABLE IF EXISTS analytics.bronze.nb_dlt_demo")
print("dropped analytics.bronze.nb_dlt_demo")

## What this maps to in production

- **Same adapter, real pipeline:** `data_platform.uc_managed_destination.make_uc_managed_destination()` + `data_platform.dlt_ingest.run_ingestion()`, orchestrated by Dagster.
- **Identity:** in Dagster a job tagged `team: <name>` runs as that team's **ingest SA** (`sa-team-<team>-ingest`, bronze-only), so a compromised ingestion job can't touch curated data. Here in the notebook, the identity is **you** (interactive dev).
- **Governance is identical:** every path goes Spark Connect → UC credential vending; no static object-store key anywhere; UC enforces the caller's grants (verified by the platform smoke harness, checks A1 + G7).
- **To ingest a real source:** replace `nb_dlt_demo()` with any [dlt source](https://dlthub.com/docs) (SQL DB, REST API, filesystem, …); name each resource after its target table and keep the same destination.